# Spectrogram Generation

In this notebook, the previously labeled vibration time-series segments are transformed into time–frequency representations using short-time Fourier transform (STFT). The resulting spectrograms are normalized and converted into RGB images by stacking the X, Y, and Z acceleration axes. These images serve as input for the subsequent convolutional neural network (CNN) used for chatter detection.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import spectrogram
from PIL import Image
import os
from tqdm import tqdm

# In-/ Output Structure & Class-Mapping

In [3]:
DATA_ROOT = Path("../data/01_windowed_labeled")
OUTPUT_ROOT = Path("../data/02_spectrograms")

CLASS_MAP = {
    "chatter": "chatter",
    "no_chatter": "no_chatter" 
}

# Spectrogram configuration
- NPERSEG: controls frequency resolution of spectrogram
larger values → better frequency resolution, lower time resolution
- MAX_FREQ_CUT: limits analysis to relevant chatter frequency range
- DB_MIN / DB_MAX: spectrogram is converted to decibel scale --> fixed scaling for energy
- TARGET_IMG_SIZE: image size of three color spectrogram

In [4]:
NPERSEG = 2048
MAX_FREQ_CUT = 5000
DB_MIN = -80
DB_MAX = -20
EPS = 1e-12

TARGET_IMG_SIZE = (150, 100)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for c in CLASS_MAP.values():
    (OUTPUT_ROOT / c).mkdir(parents=True, exist_ok=True)

## Spectrogram generator function

In [5]:
def create_spectrogram(sig, fs):
    noverlap = int(0.75 * NPERSEG)

    f, t, Sxx = spectrogram(
        sig,
        fs=fs,
        nperseg=NPERSEG,
        noverlap=noverlap,
        mode="psd"
    )

    mask = f <= MAX_FREQ_CUT
    Sxx = Sxx[mask, :]

    Sxx_db = 10 * np.log10(Sxx + EPS)
    Sxx_db = np.clip(Sxx_db, DB_MIN, DB_MAX)

    return 1.0 - (Sxx_db - DB_MIN) / (DB_MAX - DB_MIN)

# RGB Image Conversion
This function converts the labeled vibration time-series segment into a 3-channel spectrogram image.
Combines three spectrograms into a single 3-channel image:

R = X-axis
G = Y-axis
B = Z-axis

Each vibration segment is transformed into a three-channel spectrogram image, where each channel corresponds to a different sensor axis. This encoding preserves spatial vibration directionality while enabling convolutional neural networks to learn cross-axis correlations relevant for chatter detection.

**Advantages of these representation:**
- harmonic stripes (chatter)
- broadband noise (instability)
- axis coupling effects

In [6]:
def process_npz(npz_path):
    data = np.load(npz_path)

    x = data["X"]
    y = data["Y"]
    z = data["Z"]

    label = str(data["label"])
    label_folder = CLASS_MAP.get(label, "unknown")

    fs = 50000  # fallback

    specs = [
        create_spectrogram(x, fs),
        create_spectrogram(y, fs),
        create_spectrogram(z, fs)
    ]

    h, w = specs[0].shape
    rgb = np.stack(specs, axis=-1)

    rgb = np.clip(rgb, 0, 1)
    rgb = np.flipud(rgb)

    img = Image.fromarray((rgb * 255).astype(np.uint8))
    img = img.resize(TARGET_IMG_SIZE)

    experiment = npz_path.parent.parent.name

    out_name = f"{experiment}_{npz_path.stem}_{label}.png"

    out_path = OUTPUT_ROOT / label_folder / out_name

    img.save(out_path)


In [7]:

npz_files = list(DATA_ROOT.rglob("*.npz"))

print(f"Found {len(npz_files)} npz files")

for npz_file in tqdm(npz_files):
    try:
        process_npz(npz_file)
    except Exception as e:
        print(f"Error in {npz_file}: {e}")

Found 325 npz files


100%|██████████| 325/325 [01:29<00:00,  3.63it/s]
